# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a comprehensive guide for loading and exploring the dataset [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://doi.org/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Do not treat as dict/list

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` fields.

### List all available record sets:

In [ ]:
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"@id: {rs.id}\n  name: {rs.name}")
    if hasattr(rs, 'description') and rs.description:
        print(f"  description: {rs.description}")
    print("  fields:")
    for f in rs.fields:
        print(f"    - @id: {f.id}, name: {f.name}, dataType: {f.data_type}")
    print("")

### Preview records from the main record set
- Select the primary record set `@id` (from above).

In [ ]:
# For this dataset, main data is usually in the first record set
if len(record_sets) > 0:
    main_record_set_id = record_sets[0].id
    print(f"Previewing records from record set: {main_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
        print(record)
        if i >= 2:  # Show just first 3 records
            break
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for analysis. All record set and field references use their `@id`.

In [ ]:
# Extract all record sets into dataframes using their @id
dataframes = dict()

for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded DataFrame for record set {rs.id} ({rs.name}), shape: {df.shape}")
    print(f"Fields ([email protected]): {list(df.columns)[:5]}{' ...' if len(df.columns)>5 else ''}\n")

# Select the record set to analyze further
record_set_id = main_record_set_id
df_main = dataframes[record_set_id] if record_set_id in dataframes else None
# Display column @ids
if df_main is not None:
    print("Sample columns (@id):", df_main.columns.tolist())
    display(df_main.head())

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, and grouping using the [email protected] of fields.

First, enumerate all numeric fields in the main record set.

In [ ]:
# List numeric (Float or Integer) fields for analysis
main_rs = next((rs for rs in record_sets if rs.id == record_set_id), None)
numeric_fields = [f.id for f in main_rs.fields if f.data_type in ('schema:Float', 'schema:Integer', 'schema:Number')]
print("Numeric fields in main record set:", numeric_fields)
if len(numeric_fields) == 0:
    raise RuntimeError("No numeric fields found. Please check dataset schema.")

In [ ]:
# Example: use the first available numeric field
numeric_field = numeric_fields[0]  # e.g., '@row/coverage_percent' or similar
print(f"Selected numeric field: {numeric_field}")

# Coerce to numeric (errors='coerce' sets invalids to NaN)
df_main[numeric_field] = pd.to_numeric(df_main[numeric_field], errors='coerce')
threshold = df_main[numeric_field].mean() if not np.isnan(df_main[numeric_field].mean()) else 0  # Use mean as threshold

filtered_df = df_main[df_main[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.3f} (showing up to 5):")
display(filtered_df.head())

In [ ]:
# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized '{numeric_field}' for filtered records (showing up to 5):")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

In [ ]:
# Group by a categorical field if available
# Find first string field (schema:Text etc.) that's not the numeric field
cat_fields = [f.id for f in main_rs.fields if f.data_type in ('schema:Text', 'schema:String') and f.id != numeric_field]
if cat_fields:
    group_field = cat_fields[0]
    print(f"Grouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(grouped_df.head())
else:
    print("No categorical (group) field found to group by.")

## 5. Visualization
Visualize distribution and relationships using fields referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df_main[numeric_field].dropna(), kde=True, bins=30)
plt.title(f"Distribution of '{numeric_field}'")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot grouped by the group_field (if it exists)
if 'group_field' in locals() and group_field in df_main.columns:
    plt.figure(figsize=(12, 5))
    # Only show most frequent groups for clarity
    top_categories = df_main[group_field].value_counts().index[:5]
    sns.boxplot(y=df_main[df_main[group_field].isin(top_categories)][numeric_field],
                x=df_main[df_main[group_field].isin(top_categories)][group_field])
    plt.title(f"'{numeric_field}' grouped by '{group_field}'")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()
else:
    print("No categorical group field for boxplot.")

## 6. Conclusion
This notebook demonstrated loading a Croissant-structured dataset with `mlcroissant`, referencing all structural elements using their `@id`, and conducting basic exploratory data analysis and visualization.

**Key observations:**
- The data structure (record sets and fields) is described in the Croissant schema, with each field accessible by its unique `@id`.
- Filtering, normalization, and grouping operations can be performed directly using these standard identifiers.
- Visualizing numeric attributes provides insight into protein abundance and related mass spectrometry results.

Further analysis may include more advanced normalization, imputation of missing values, or domain-specific interpretations depending on research aims.